# The Price Is Right - Extension

This notebook enhances week 8 project by integrating an additional agent that will send a message to another person e.g. *your brother*, to review the price and if they agree with the discount, the agent will further request to the shop to deliver the item and send an invoice.

In [ ]:
# Imports
import sys
import os
from typing import override, List, Optional

# Change working directory to incorporate week 8 project.
notebook_dir = os.getcwd()
week8_dir = os.path.join(notebook_dir, '..', '..')
os.chdir(week8_dir)
print(f"Working directory: {os.getcwd()}")

import logging

Price review agent

In [ ]:
from agents.agent import Agent
from litellm import completion
import requests

pushover_url = "https://api.pushover.net/1/messages.json"

class PriceReviewAgent(Agent):
    name = "Price Review Agent"
    color = Agent.RED
    MODEL = "claude-sonnet-4-5"
    REVIEW_RESPONSE = "I agree with the price"

    def __init__(self):
        """
        This agent will do push notifications via Pushover,
        """
        self.log(f"{self.name} is initializing")
        self.pushover_user = os.getenv("PUSHOVER_USER", "your-pushover-user-if-not-using-env")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "your-pushover-user-if-not-using-env")
        self.log(f"{self.name} has initialized Pushover and Claude")
    
    def push(self, text):
        """
        Send a Push Notification using the Pushover API
        """
        self.log(f"{self.name} is sending a push notification")
        payload = {
            "user": self.pushover_user,
            "token": self.pushover_token,
            "message": text,
            "sound": "cashregister",
        }
        requests.post(pushover_url, data=payload)

    def craft_review_request_message(
        self, description: str, deal_price: float, estimated_true_value: float
    ) -> str:
        """
        Craft a review request message to your brother to review the offer and if they agree with the discount.
        """
        self.log(f"{self.name} is crafting a review request message for the deal")
        user_prompt = "Please summarize the following deal in 4-5 sentences to be sent as a review request message to my brother. They should reply to the message with their approval or disapproval.\n"
        user_prompt += f"Item Description: {description}\nOffered Price: {deal_price}\nEstimated true value: {estimated_true_value}"
        user_prompt += "\n\nRespond only with the 4-5 sentence message which will be sent to my brother asking them to review the offer."
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content
    
    def send_review_request(self, description: str, deal_price: float, estimated_true_value: float):
        """
        Send a review request message.
        """
        self.log(f"{self.name} is sending a review request message")
        message = self.craft_review_request_message(description, deal_price, estimated_true_value)
        self.push(message)

    def get_review_response(self) -> str:
        """
        Get the review response from the reveiwer.
        """
        self.log(f"{self.name} is getting the review response")
        return self.REVIEW_RESPONSE
    
    def analyse_review_response(self, message: str) -> bool:
        """
        Analyse the review response and return a boolean value
        """
        self.log(f"{self.name} is analysing the review response")
        user_prompt = f"Please analyse the following message from the reveiwer and respond with 'true' indicating if they agree with the discount. Respond with an empty string if they do not agree with the discount.\n"
        user_prompt += f"Message: {message}"
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return True if response.choices[0].message.content == "true" else False

    def craft_purchase_order(self, deal: Deal):
        """
        Send a purchase order for the deal
        """
        self.log(f"{self.name} is sending a purchase order for the deal to the merchant.")
        user_prompt = f"Please craft a purchase order for the following deal to be sent to the merchant.\n"
        user_prompt += f"Deal: {deal}"
        response = completion(
            model=self.MODEL,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )
        return response.choices[0].message.content
    
    def send_purchase_order(self, deal: Deal):
        """
        Send a purchase order for the deal
        """
        self.log(f"{self.name} is sending a purchase order for the deal to the merchant.")
        message = self.craft_purchase_order(deal)
        self.push(message)
    
    

## Extended Planning Agent